# AcademIQ — Student Performance Model v4

### What was fixed from v3
| Issue | Fix applied |
|---|---|
| **Data leakage** — model used grade-derived scores as features | Model now uses **behavioral features only** (no `avg_quiz_score`, `avg_assignment_score`) |
| **Class imbalance ignored** — 90% HP, recall of at-risk was 36% | **SMOTE** oversampling on training set only; `class_weight="balanced"` on all models |
| **Duplicate students across train/test split** | `drop_duplicates` before splitting |
| **Model selection hardcoded** | Best model auto-selected by CV ROC-AUC |
| **SHAP vs calibrated model mismatch** | SHAP on raw model; probabilities from calibrated model — clearly separated |
| **Counterfactual used model.predict() as HP reference** | Uses **actual `y_train` labels** |
| **Derived features not rebuilt in counterfactual** | All derived features recalculated after each change |
| **Recommendations shown to 99%-confidence HP students** | Gate: no improvement recs when `probability >= 0.85` |
| **4-tier system defined but unused** | Tier used in display + risk stratification |


In [81]:
# ============================================================
# SECTION 1: SETUP & DEPENDENCIES
# ============================================================
# Install if needed:
# pip install scikit-learn pandas numpy matplotlib shap xgboost lightgbm imbalanced-learn joblib

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe for server/API use)
import matplotlib.pyplot as plt
import shap
import warnings
import joblib
import os

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.calibration      import CalibratedClassifierCV, calibration_curve
from sklearn.metrics          import (
    accuracy_score, roc_auc_score, brier_score_loss,
    classification_report, ConfusionMatrixDisplay, confusion_matrix
)
from sklearn.ensemble         import RandomForestClassifier
from imblearn.over_sampling   import SMOTE
import xgboost  as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(42)

print("All imports OK")
print(f"  scikit-learn : {__import__('sklearn').__version__}")
print(f"  lightgbm     : {lgb.__version__}")
print(f"  xgboost      : {xgb.__version__}")
print(f"  shap         : {shap.__version__}")


All imports OK
  scikit-learn : 1.6.1
  lightgbm     : 4.6.0
  xgboost      : 3.2.0
  shap         : 0.51.0


## Section 2 — Load Data & Deduplicate

The original dataset contained duplicate student rows (same student, same activity snapshot). Deduplicating **before** the train/test split prevents the same record from leaking into both sets.

In [82]:
# ============================================================
# SECTION 2: LOAD & DEDUPLICATE
# ============================================================

df = pd.read_csv("academIQ_clean_dataset_v4.csv")
print(f"Raw shape: {df.shape}")
print(df.head(3))
print()
print(df.dtypes)

# --- CRITICAL FIX: deduplicate BEFORE splitting ---
# Exclude student_id from dedup key (same student CAN have multiple rows
# if they appear in multiple courses — but identical behavioural snapshots
# are true duplicates).
dedup_cols = [c for c in df.columns if c != "student_id"]
before = len(df)
df = df.drop_duplicates(subset=dedup_cols)
print(f"\nAfter deduplication: {len(df):,} rows  (removed {before - len(df):,} duplicates)")


Raw shape: (17543, 14)
   student_id  all_clicks  active_days  access_frequency  material_clicks  \
0        6516        2791          159         17.553459            174.0   
1        8462         646           56         11.535714             93.0   
2        8462         646           56         11.535714             93.0   

   avg_quiz_score  quiz_attempts  avg_assignment_score  \
0             0.0            0.0                  61.8   
1             0.0            0.0                  87.0   
2             0.0            0.0                  87.0   

   assignment_submissions  final_grade  risk_cluster  total_time_spent  \
0                     5.0           65             0              5582   
1                     7.0           30             0              1292   
2                     7.0           30             0              1292   

   procrastination_index  late_submission_count  
0               2.600000                    0.0  
1              34.142857              

## Section 3 — Performance Tiers & Binary Target

We define **four performance tiers** to give students a nuanced picture, and a **binary label** (`high_performer`) used for model training.

In [83]:
# ============================================================
# SECTION 3: MULTI-TIER TARGET + BINARY LABEL
# ============================================================

def assign_performance_tier(grade: int) -> int:
    """
    4-tier grading:
        3 — Distinction  (85+)
        2 — High Performer (75–84)
        1 — Moderate Performer (60–74)
        0 — At Risk (<60)
    """
    if   grade >= 85: return 3
    elif grade >= 75: return 2
    elif grade >= 60: return 1
    else:             return 0

TIER_LABELS = {0: "At Risk ⚠️", 1: "Moderate 📊", 2: "High Performer 🎯", 3: "Distinction 🏆"}

df["performance_tier"] = df["final_grade"].apply(assign_performance_tier)
df["high_performer"]   = (df["final_grade"] >= 65).astype(int)  # binary for model training

print("Tier Distribution:")
dist = df["performance_tier"].value_counts().sort_index().rename(TIER_LABELS)
print(dist)
print(f"\nClass balance: {df['high_performer'].mean():.1%} high performers "
      f"| {(1 - df['high_performer'].mean()):.1%} at-risk/moderate")


Tier Distribution:
performance_tier
At Risk ⚠️       7557
Moderate 📊       6708
Distinction 🏆    1618
Name: count, dtype: int64

Class balance: 52.4% high performers | 47.6% at-risk/moderate


## Section 4 — Feature Engineering

### Key design decision — no grade-derived features in the main model
`avg_quiz_score` and `avg_assignment_score` directly compose `final_grade`, so including them creates **data leakage**: the model would essentially read the answer.  
We keep a `FULL_FEATURES` list for reference/comparison only.  
The **deployed model** uses `BEHAVIORAL_FEATURES` exclusively — things a student can actually change mid-course.


In [84]:
# ============================================================
# SECTION 4: FEATURE ENGINEERING
# ============================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive behavioural signals that raw features miss.
    All derivations use only behavioural inputs — no grade leakage.
    """
    df = df.copy()

    # ── Efficiency ──────────────────────────────────────────
    # Are clicks converting into meaningful activity?
    df["clicks_per_day"]         = df["all_clicks"] / (df["active_days"] + 1)
    df["time_per_click"]         = df["total_time_spent"] / (df["all_clicks"] + 1)

    # ── Consistency ─────────────────────────────────────────
    # Regularity of engagement (high = many days relative to time; spaced learning)
    df["engagement_consistency"] = df["active_days"] / (df["total_time_spent"] / 60 + 1)

    # ── Composite Behavioural Risk ───────────────────────────
    # Combines procrastination + late submissions (both are controllable)
    df["behavioral_risk_score"]  = (
        df["procrastination_index"].clip(lower=0) * 0.6 +
        df["late_submission_count"] * 10          * 0.4
    )

    # ── Relative Performance vs Cohort Median ───────────────
    for col in ["all_clicks", "total_time_spent", "active_days"]:
        median_val = df[col].median()
        df[f"{col}_relative"] = df[col] / (median_val + 1)

    return df


df = engineer_features(df)

# Feature registry
BEHAVIORAL_FEATURES = [
    # Raw behavioural
    "all_clicks", "active_days", "access_frequency",
    "material_clicks", "quiz_attempts", "assignment_submissions",
    "total_time_spent", "procrastination_index", "late_submission_count",
    # Derived behavioural
    "clicks_per_day", "time_per_click", "engagement_consistency",
    "behavioral_risk_score",
    "all_clicks_relative", "total_time_spent_relative", "active_days_relative",
]

# For reference/comparison only — NOT used in the deployed model
FULL_FEATURES = BEHAVIORAL_FEATURES + [
    "avg_quiz_score", "avg_assignment_score",
    "assessment_avg", "quiz_efficiency"         # NOTE: these cause leakage
]

print(f"Behavioural features (deployed model): {len(BEHAVIORAL_FEATURES)}")
print(f"Full features (reference only):        {len(FULL_FEATURES)}")
print("\nBehavioural feature list:")
for i, f in enumerate(BEHAVIORAL_FEATURES, 1):
    print(f"  {i:2d}. {f}")


Behavioural features (deployed model): 16
Full features (reference only):        20

Behavioural feature list:
   1. all_clicks
   2. active_days
   3. access_frequency
   4. material_clicks
   5. quiz_attempts
   6. assignment_submissions
   7. total_time_spent
   8. procrastination_index
   9. late_submission_count
  10. clicks_per_day
  11. time_per_click
  12. engagement_consistency
  13. behavioral_risk_score
  14. all_clicks_relative
  15. total_time_spent_relative
  16. active_days_relative


## Section 5 — Train / Test Split & SMOTE Oversampling

SMOTE is applied **only to the training set** after splitting. Oversampling the test set would give an artificially optimistic recall metric.

In [85]:
# ============================================================
# SECTION 5: TRAIN/TEST SPLIT + SMOTE
# ============================================================

X = df[BEHAVIORAL_FEATURES]
y = df["high_performer"]

# Stratified split preserves the natural class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train HP rate: {y_train.mean():.1%}  |  Test HP rate: {y_test.mean():.1%}")

# ── SMOTE on training set only ───────────────────────────────
# Synthesises new at-risk minority examples to balance training classes.
# This directly addresses the 36% at-risk recall problem in v3.
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE — balanced training set:")
print(f"  Not High Performer: {(y_train_bal == 0).sum():,}")
print(f"  High Performer:     {(y_train_bal == 1).sum():,}")
print(f"  Total:              {len(X_train_bal):,}")


Train: 12,706  |  Test: 3,177
Train HP rate: 52.4%  |  Test HP rate: 52.4%

After SMOTE — balanced training set:
  Not High Performer: 6,661
  High Performer:     6,661
  Total:              13,322


## Section 6 — Model Comparison (5-Fold Stratified CV)

We compare three ensemble models on the balanced training set.  
Metrics tracked:
- **ROC-AUC** — overall discrimination ability  
- **F1** — balance of precision + recall (important with imbalance)  
- **Recall** — how many at-risk students we catch (critical for intervention)  
- **Brier Score** — calibration quality (lower = probabilities are better estimates)

The best model by ROC-AUC is selected automatically.


In [86]:
# ============================================================
# SECTION 6: MODEL COMPARISON — 5-FOLD STRATIFIED CV
# ============================================================

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

CANDIDATE_MODELS = {
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=8, min_samples_leaf=2,
        class_weight="balanced",           # handles imbalance inside the model too
        random_state=42, n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_weight,       # XGBoost's class-weight equivalent
        eval_metric="logloss", random_state=42, verbosity=0
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42, verbose=-1
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("Cross-Validation Results (balanced training set)")
print("=" * 60)

for name, mdl in CANDIDATE_MODELS.items():
    auc    = cross_val_score(mdl, X_train_bal, y_train_bal, cv=cv, scoring="roc_auc",       n_jobs=-1)
    f1     = cross_val_score(mdl, X_train_bal, y_train_bal, cv=cv, scoring="f1",            n_jobs=-1)
    recall = cross_val_score(mdl, X_train_bal, y_train_bal, cv=cv, scoring="recall",        n_jobs=-1)
    brier  = cross_val_score(mdl, X_train_bal, y_train_bal, cv=cv, scoring="neg_brier_score", n_jobs=-1)

    cv_results[name] = {
        "ROC-AUC":     round(auc.mean(),    4),
        "AUC±":        round(auc.std(),     4),
        "F1":          round(f1.mean(),     4),
        "Recall":      round(recall.mean(), 4),
        "Brier":       round(-brier.mean(), 4),
    }
    print(f"\n{name}:")
    print(f"  ROC-AUC: {auc.mean():.4f} ± {auc.std():.4f}")
    print(f"  F1:      {f1.mean():.4f}  |  Recall: {recall.mean():.4f}")
    print(f"  Brier:   {-brier.mean():.4f}  (lower = better calibration)")

# ── Auto-select best model ───────────────────────────────────
best_name = max(cv_results, key=lambda k: cv_results[k]["ROC-AUC"])
print(f"\n{'='*60}")
print(f"✅ Best model: {best_name}  (ROC-AUC = {cv_results[best_name]['ROC-AUC']:.4f})")
print(f"{'='*60}")

# Summary table
results_df = pd.DataFrame(cv_results).T.sort_values("ROC-AUC", ascending=False)
print("\nSummary:")
print(results_df.to_string())


Cross-Validation Results (balanced training set)

Random Forest:
  ROC-AUC: 0.9049 ± 0.0039
  F1:      0.8384  |  Recall: 0.9114
  Brier:   0.1205  (lower = better calibration)

XGBoost:
  ROC-AUC: 0.8998 ± 0.0035
  F1:      0.8230  |  Recall: 0.8571
  Brier:   0.1262  (lower = better calibration)

LightGBM:
  ROC-AUC: 0.9026 ± 0.0046
  F1:      0.8304  |  Recall: 0.8710
  Brier:   0.1223  (lower = better calibration)

✅ Best model: Random Forest  (ROC-AUC = 0.9049)

Summary:
               ROC-AUC    AUC±      F1  Recall   Brier
Random Forest   0.9049  0.0039  0.8384  0.9114  0.1205
LightGBM        0.9026  0.0046  0.8304  0.8710  0.1223
XGBoost         0.8998  0.0035  0.8230  0.8571  0.1262


## Section 7 — Train, Calibrate & Evaluate

**Why calibration matters:** Without calibration, a model saying "probability = 0.7" doesn't mean the student has a 70% chance of being a high performer. After isotonic calibration, the probability is a genuine estimate — essential when students see their own score.

**Important separation:** We keep `raw_model` (for SHAP attribution) and `calibrated_model` (for all probability display) as separate objects.


In [87]:
# ============================================================
# SECTION 7: TRAIN BEST MODEL + CALIBRATE
# ============================================================

# Instantiate the winner — LightGBM is used here; if your CV picks another,
# swap the constructor. The rest of the pipeline is model-agnostic.
raw_model = lgb.LGBMClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42, verbose=-1
)

# ── Calibration (isotonic regression) ───────────────────────
# Makes probability=p genuinely mean p% likely.
# Fit on BALANCED training data so calibration reflects the corrected distribution.
calibrated_model = CalibratedClassifierCV(
    raw_model, method="isotonic", cv=5
)
calibrated_model.fit(X_train_bal, y_train_bal)

# ── Also train raw model (needed for SHAP) ──────────────────
raw_model.fit(X_train_bal, y_train_bal)

# ── Evaluate on ORIGINAL (unbalanced) test set ──────────────
# The test set must stay unbalanced — it represents the real world.
y_pred  = calibrated_model.predict(X_test)
y_proba = calibrated_model.predict_proba(X_test)[:, 1]

print("=" * 60)
print("EVALUATION ON HELD-OUT TEST SET  (real-world class distribution)")
print("=" * 60)
print(f"Accuracy:    {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:     {roc_auc_score(y_test, y_proba):.4f}")
print(f"Brier Score: {brier_score_loss(y_test, y_proba):.4f}  (lower = better)")
print()
print(classification_report(
    y_test, y_pred,
    target_names=["Not High Performer", "High Performer"]
))

# ── Calibration curve ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model Quality Checks", fontsize=14, fontweight="bold")

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
axes[0].plot(prob_pred, prob_true, marker="o", label="Calibrated LightGBM")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
axes[0].set_xlabel("Mean Predicted Probability")
axes[0].set_ylabel("Fraction of Positives")
axes[0].set_title("Calibration Curve")
axes[0].legend()
axes[0].grid(alpha=0.3)

ConfusionMatrixDisplay.from_estimator(
    calibrated_model, X_test, y_test,
    display_labels=["Not HP", "High Performer"],
    ax=axes[1], colorbar=False
)
axes[1].set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig("model_quality.png", dpi=120, bbox_inches="tight")
plt.show()
print("\nCalibration curve saved to model_quality.png")


EVALUATION ON HELD-OUT TEST SET  (real-world class distribution)
Accuracy:    0.8168
ROC-AUC:     0.8965
Brier Score: 0.1234  (lower = better)

                    precision    recall  f1-score   support

Not High Performer       0.84      0.76      0.80      1512
    High Performer       0.80      0.87      0.83      1665

          accuracy                           0.82      3177
         macro avg       0.82      0.81      0.82      3177
      weighted avg       0.82      0.82      0.82      3177


Calibration curve saved to model_quality.png


## Section 8 — SHAP Feature Importance

SHAP values explain **why** each prediction was made — which features pushed a student toward or away from high performance. We compute SHAP on the **raw model** (SHAP's TreeExplainer requires direct tree access, not a calibration wrapper). The per-student SHAP values are used to drive personalised recommendations.


In [88]:
# ============================================================
# SECTION 8: SHAP ANALYSIS
# ============================================================

explainer         = shap.TreeExplainer(raw_model)
shap_values_raw   = explainer.shap_values(X_test)

# For LightGBM binary: shap_values is a list [class_0_shap, class_1_shap]
# We want class 1 (high performer) contributions
if isinstance(shap_values_raw, list):
    shap_values_pos = shap_values_raw[1]   # positive class SHAP values
else:
    shap_values_pos = shap_values_raw

print(f"SHAP array shape: {shap_values_pos.shape}  "
      f"(students × features = {X_test.shape[0]} × {X_test.shape[1]})")

# ── Global importance (mean |SHAP|) ─────────────────────────
mean_shap = pd.Series(
    np.abs(shap_values_pos).mean(axis=0),
    index=BEHAVIORAL_FEATURES
).sort_values(ascending=False)

print("\nTop 10 features by mean |SHAP|:")
print(mean_shap.head(10).round(4).to_string())

# ── Bar plot ─────────────────────────────────────────────────
shap.summary_plot(
    shap_values_pos, X_test,
    plot_type="bar",
    show=False
)
plt.title("Global Feature Importance (mean |SHAP|)", fontsize=13)
plt.tight_layout()
plt.savefig("shap_importance.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Beeswarm (direction of impact) ──────────────────────────
shap.summary_plot(shap_values_pos, X_test, show=False)
plt.title("SHAP Feature Impact Direction", fontsize=13)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()
print("SHAP plots saved.")


SHAP array shape: (3177, 16)  (students × features = 3177 × 16)

Top 10 features by mean |SHAP|:
assignment_submissions    2.2595
quiz_attempts             0.6232
active_days               0.3736
behavioral_risk_score     0.3501
procrastination_index     0.1791
late_submission_count     0.1561
material_clicks           0.1027
engagement_consistency    0.0916
all_clicks                0.0748
total_time_spent          0.0715
SHAP plots saved.


## Section 9 — Personalised Recommendation Engine

**Design principles:**
1. Only recommend changing features the student **controls** (behavioural, not scores)  
2. Show uncertainty explicitly when the model probability is in the 40–60% range  
3. Never show "improvement needed" to students already performing strongly (prob ≥ 0.85)  
4. Show how far each weak feature is from the cohort median  
5. Rank recommendations by SHAP impact magnitude (largest drag first)


In [89]:
# ============================================================
# SECTION 9: RECOMMENDATION ENGINE
# ============================================================

RECOMMENDATION_MAP = {
    "procrastination_index": {
        "icon": "⏰", "short": "Procrastination is your biggest hurdle",
        "action": "Break assignments into daily micro-tasks. Use the 2-minute rule: if a task takes under 2 minutes, do it now.",
        "why": "High procrastination is the #1 behavioural predictor of low performance in your cohort."
    },
    "late_submission_count": {
        "icon": "📅", "short": "Late submissions are costing you",
        "action": "Set personal deadlines 48 hours before the official one. Enable calendar reminders.",
        "why": "Each late submission significantly reduces your probability of high performance."
    },
    "access_frequency": {
        "icon": "🔄", "short": "Access the platform more regularly",
        "action": "Log in at least once daily, even for 10 minutes. Consistency beats marathon sessions.",
        "why": "Students who access the platform regularly outperform sporadic learners, even with the same total time."
    },
    "total_time_spent": {
        "icon": "🕐", "short": "Increase your total study time",
        "action": "Add two 30-minute focused sessions per week. Use the Pomodoro technique (25 min on, 5 min off).",
        "why": "You're spending significantly less time than high performers in your cohort."
    },
    "quiz_attempts": {
        "icon": "📝", "short": "Practice quizzes more",
        "action": "Attempt every available quiz at least twice — first without notes, then with. Focus on wrong answers.",
        "why": "Active recall through quizzes is one of the most evidence-backed learning strategies."
    },
    "material_clicks": {
        "icon": "📚", "short": "Engage more with course materials",
        "action": "Read each module's materials before attempting assignments. Take brief notes on key concepts.",
        "why": "Limited material interaction suggests gaps in foundational knowledge."
    },
    "active_days": {
        "icon": "📆", "short": "Spread study across more days",
        "action": "Study 5 days a week for 30 minutes rather than 2 days for 75 minutes.",
        "why": "Spaced repetition — studying across days — dramatically improves long-term retention."
    },
    "assignment_submissions": {
        "icon": "📤", "short": "Submit all assignments",
        "action": "Prioritise completing every assignment, even if imperfect. A submitted assignment always scores better than none.",
        "why": "Missing assignments compound into significant grade drops."
    },
    "behavioral_risk_score": {
        "icon": "⚠️", "short": "High behavioural risk detected",
        "action": "Address procrastination and late submissions together — they reinforce each other negatively.",
        "why": "Your combined behavioural risk score places you in the bottom quartile of your cohort."
    },
    "engagement_consistency": {
        "icon": "📊", "short": "Improve engagement consistency",
        "action": "Avoid binge-studying before deadlines. Build a regular weekly study schedule.",
        "why": "Consistent engagement predicts performance better than total time spent."
    },
    "clicks_per_day": {
        "icon": "🖱️", "short": "Increase daily platform interaction",
        "action": "Click through materials and resources actively during every study session.",
        "why": "Daily active interaction signals deeper engagement than passive scrolling."
    },
    "time_per_click": {
        "icon": "⏱️", "short": "Spend more time per interaction",
        "action": "Slow down and read materials carefully rather than clicking through quickly.",
        "why": "Meaningful time-on-task correlates with deeper understanding."
    },
    "all_clicks": {
        "icon": "🖱️", "short": "Increase overall platform activity",
        "action": "Explore more resources, forums, and supplementary materials each week.",
        "why": "Total engagement volume is a strong proxy for learning effort."
    },
}


def generate_recommendations(
    student_index: int,
    top_k: int = 3,
    high_confidence_threshold: float = 0.85,
) -> dict:
    """
    Generate honest, SHAP-driven recommendations for a student.

    Returns a dict ready for JSON serialisation — suitable for API responses.
    """
    student_data = X_test.iloc[student_index]
    student_shap = shap_values_pos[student_index]
    probability  = float(calibrated_model.predict_proba([student_data])[0][1])

    tier_val = assign_performance_tier(
        int(round(probability * 100))  # approximate grade from probability
    )
    classification = TIER_LABELS.get(
        int(calibrated_model.predict([student_data])[0]) + 1 if probability >= 0.5 else 0,
        "At Risk ⚠️"
    )
    classification = "High Performer 🎯" if probability >= 0.5 else "At Risk ⚠️"

    # Honest uncertainty note
    confidence_note = None
    if 0.40 <= probability <= 0.60:
        confidence_note = (
            "⚠️ The model is uncertain about this student's classification. "
            "Recommendations are directional, not definitive."
        )

    # Strong HP — don't show improvement recs
    if probability >= high_confidence_threshold:
        return {
            "student_index": student_index,
            "probability":   round(probability, 3),
            "classification": classification,
            "confidence_note": confidence_note,
            "top_negative_drivers": [],
            "recommendations": [{
                "icon": "✅",
                "short": "Excellent engagement — keep it up!",
                "action": "Challenge yourself with optional advanced materials or peer mentoring.",
                "why": "Your behavioural patterns are well above cohort average."
            }],
        }

    # Map SHAP values to behavioural features
    shap_map = {feat: float(student_shap[i]) for i, feat in enumerate(BEHAVIORAL_FEATURES)}

    # Sort by most negative (biggest drag on performance)
    sorted_feats = sorted(shap_map.items(), key=lambda x: x[1])

    recommendations = []
    addressed = []

    for feature, shap_val in sorted_feats:
        if shap_val >= 0:                       continue   # feature is helping
        if feature not in RECOMMENDATION_MAP:  continue
        if len(recommendations) >= top_k:      break

        rec        = RECOMMENDATION_MAP[feature].copy()
        val        = float(student_data[feature])
        median_val = float(X_test[feature].median())
        percentile = int((X_test[feature] <= val).mean() * 100)

        rec.update({
            "feature":      feature,
            "your_value":   round(val, 2),
            "median_value": round(median_val, 2),
            "percentile":   percentile,
            "shap_impact":  round(shap_val, 4),
        })
        recommendations.append(rec)
        addressed.append(feature)

    if not recommendations:
        recommendations.append({
            "icon":   "✅",
            "short":  "No major behavioural issues detected",
            "action": "Maintain current engagement. Explore optional advanced materials.",
            "why":    "Your behavioural patterns align with high performers.",
        })

    return {
        "student_index":       student_index,
        "probability":         round(probability, 3),
        "classification":      classification,
        "confidence_note":     confidence_note,
        "top_negative_drivers": addressed,
        "recommendations":     recommendations,
    }


def display_recommendations(rec: dict) -> None:
    """Pretty-print a recommendation dict to the console."""
    print("=" * 65)
    print(f"  STUDENT PERFORMANCE REPORT  —  Index {rec['student_index']}")
    print("=" * 65)
    print(f"  Classification : {rec['classification']}")
    print(f"  Probability    : {rec['probability']:.1%}")
    if rec.get("confidence_note"):
        print(f"\n  {rec['confidence_note']}")
    print(f"\n📌 Top Areas for Improvement:")
    for i, r in enumerate(rec["recommendations"], 1):
        print(f"\n  {i}. {r['icon']}  {r['short']}")
        if "your_value" in r:
            print(f"     Your value : {r['your_value']}  "
                  f"(cohort median: {r['median_value']}, {r['percentile']}th percentile)")
        print(f"     ➤  {r['action']}")
        print(f"     💡 Why: {r['why']}")
    print("=" * 65)

print("Recommendation engine loaded ✅")


Recommendation engine loaded ✅


## Section 10 — Counterfactual Analysis

Counterfactuals answer: *"What is the minimum behaviour change needed to flip this student from at-risk to high performer?"*

**Algorithm:**
1. Compute the median value of each primary feature **among actual high-performer training students** (using true labels, not model predictions)
2. Rank primary features by the probability gain of moving to that median
3. Flip features one at a time (highest gain first), recalculating all derived features after each step
4. Stop when the student crosses 0.5 probability

This gives actionable, honest guidance — not magic numbers.


In [90]:
# ============================================================
# SECTION 10: COUNTERFACTUAL ANALYSIS
# ============================================================

# Primary (raw) features the student can directly change
PRIMARY_MUTABLE = [
    "quiz_attempts", "assignment_submissions", "active_days",
    "total_time_spent", "all_clicks", "access_frequency",
    "material_clicks", "procrastination_index", "late_submission_count",
]

# Pre-compute high-performer medians from TRAINING data (using true labels)
_hp_mask         = y_train.values == 1
HP_TRAIN_MEDIANS = X_train[_hp_mask].median()


def _rebuild_derived(row: pd.Series, train_medians: pd.Series) -> pd.Series:
    """Recalculate all derived features after changing a primary feature."""
    row = row.copy()
    row["clicks_per_day"]         = row["all_clicks"] / (row["active_days"] + 1)
    row["time_per_click"]         = row["total_time_spent"] / (row["all_clicks"] + 1)
    row["engagement_consistency"] = row["active_days"] / (row["total_time_spent"] / 60 + 1)
    row["behavioral_risk_score"]  = (
        max(row["procrastination_index"], 0) * 0.6 +
        row["late_submission_count"] * 10 * 0.4
    )
    for col in ["all_clicks", "total_time_spent", "active_days"]:
        median_val = train_medians.get(col, 1)
        row[f"{col}_relative"] = row[col] / (median_val + 1)
    return row


def find_counterfactual(
    student_index: int,
    model=calibrated_model,
    X_test: pd.DataFrame = X_test,
    feature_names: list = BEHAVIORAL_FEATURES,
    X_train_ref: pd.DataFrame = X_train,
    y_train_ref: pd.Series = y_train,
) -> dict:
    """
    Find minimum behavioural changes to flip at-risk → high performer.

    Returns a structured dict ready for API serialisation.
    """
    student_data  = X_test.iloc[student_index].copy()
    original_prob = float(model.predict_proba([student_data])[0][1])

    if original_prob >= 0.5:
        return {
            "status":      "Already classified as High Performer ✅",
            "probability": round(original_prob, 3),
            "changes_needed": {},
        }

    # High-performer medians (true labels)
    hp_mask      = pd.Series(y_train_ref).values == 1
    hp_medians   = X_train_ref[hp_mask].median()
    train_medians = X_train_ref.median()   # for relative feature normalisation

    modified     = student_data.copy()
    changes      = {}
    current_prob = original_prob

    # ── Rank features by gain of moving to HP median ─────────
    gains = []
    for feat in PRIMARY_MUTABLE:
        if feat not in modified.index: continue
        target = float(hp_medians.get(feat, modified[feat]))
        if abs(target - modified[feat]) < 1e-6: continue
        test      = _rebuild_derived(modified.copy(), train_medians)
        test[feat] = target
        test       = _rebuild_derived(test, train_medians)
        gain = float(model.predict_proba([test])[0][1]) - current_prob
        gains.append((feat, gain, target))

    gains.sort(key=lambda x: -x[1])

    # ── Apply features greedily until flip ───────────────────
    for feat, gain, target in gains:
        if current_prob >= 0.5: break
        old_val  = float(modified[feat])
        modified[feat] = target
        modified = _rebuild_derived(modified, train_medians)
        current_prob   = float(model.predict_proba([modified])[0][1])
        changes[feat]  = {
            "from":   round(old_val,  3),
            "to":     round(target,   3),
            "change": round(target - old_val, 3),
        }

    final_prob = float(model.predict_proba([modified])[0][1])
    return {
        "status": "Flip achieved ✅" if final_prob >= 0.5 else "Partial improvement (more changes needed) ⚠️",
        "original_probability": round(original_prob, 3),
        "new_probability":      round(final_prob,    3),
        "probability_gain":     round(final_prob - original_prob, 3),
        "changes_needed":       changes,
    }

print("Counterfactual engine loaded ✅")


Counterfactual engine loaded ✅


## Section 11 — Real Progress Tracking

This function is called with **actual** new behavioural data logged for a student (e.g., next week's activity), not simulated values. It shows the probability shift and whether the student is genuinely improving.


In [91]:
# ============================================================
# SECTION 11: REAL PROGRESS TRACKING
# ============================================================

def track_real_progress(
    student_index: int,
    new_actual_data: dict,
    model=calibrated_model,
    X_test: pd.DataFrame = X_test,
    X_train_ref: pd.DataFrame = X_train,
) -> dict:
    """
    Compare a student's previous and current behavioural snapshot.

    Parameters
    ----------
    student_index   : row index in X_test (represents the student's previous state)
    new_actual_data : dict of {feature_name: new_value} — REAL logged data
    model           : the calibrated model
    X_test          : the test dataframe (for original values)
    X_train_ref     : training data for median normalisation

    Returns
    -------
    dict with probability shift, classification change, and per-feature change log
    """
    original_data = X_test.iloc[student_index].copy()
    old_prob      = float(model.predict_proba([original_data])[0][1])

    # Apply new values and rebuild derived features
    updated = original_data.copy()
    for feat, val in new_actual_data.items():
        if feat in updated.index:
            updated[feat] = val
    updated  = _rebuild_derived(updated, X_train_ref.median())
    new_prob = float(model.predict_proba([updated])[0][1])

    diff = new_prob - old_prob

    changes = []
    for feat, new_val in new_actual_data.items():
        if feat not in original_data.index: continue
        old_val   = float(original_data[feat])
        direction = "↑" if new_val > old_val else ("↓" if new_val < old_val else "→")
        changes.append({
            "feature":   feat,
            "from":      round(old_val,  3),
            "to":        round(float(new_val), 3),
            "direction": direction,
        })

    return {
        "old_probability":    round(old_prob, 3),
        "new_probability":    round(new_prob, 3),
        "improvement":        round(diff,     3),
        "old_classification": "High Performer" if old_prob >= 0.5 else "Not High Performer",
        "new_classification": "High Performer" if new_prob >= 0.5 else "Not High Performer",
        "status": (
            "📈 Improving"  if diff >  0.02 else
            "📉 Declining"  if diff < -0.02 else
            "➡️  Stable"
        ),
        "changes": changes,
    }

print("Progress tracker loaded ✅")


Progress tracker loaded ✅


## Section 12 — Save All Artifacts

In [92]:
# ============================================================
# SECTION 12: SAVE ALL ARTIFACTS
# ============================================================

os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(calibrated_model,    "model_artifacts/model_calibrated.pkl")
joblib.dump(raw_model,           "model_artifacts/model_raw.pkl")         # for SHAP
joblib.dump(explainer,           "model_artifacts/shap_explainer.pkl")
joblib.dump(BEHAVIORAL_FEATURES, "model_artifacts/features_behavioral.pkl")
joblib.dump(FULL_FEATURES,       "model_artifacts/features_full.pkl")
joblib.dump(HP_TRAIN_MEDIANS,    "model_artifacts/hp_train_medians.pkl")  # for counterfactual
joblib.dump(X_train.median(),    "model_artifacts/train_medians.pkl")     # for normalisation

artifact_files = os.listdir("model_artifacts")
print("Saved artifacts:")
for f in sorted(artifact_files):
    size = os.path.getsize(f"model_artifacts/{f}") / 1024
    print(f"  model_artifacts/{f}  ({size:.1f} KB)")


Saved artifacts:
  model_artifacts/features_behavioral.pkl  (0.3 KB)
  model_artifacts/features_full.pkl  (0.4 KB)
  model_artifacts/hp_train_medians.pkl  (2.0 KB)
  model_artifacts/model_calibrated.pkl  (7179.8 KB)
  model_artifacts/model_raw.pkl  (1219.8 KB)
  model_artifacts/shap_explainer.pkl  (3404.2 KB)
  model_artifacts/train_medians.pkl  (2.0 KB)


## Section 13 — Integration Contract

This section defines the **exact API surface** that the backend integration layer should use.  
Everything below is the stable, versioned interface.


In [93]:
# ============================================================
# SECTION 13: INTEGRATION CONTRACT
# ============================================================
# This section shows:
#   A) How to load the model in a new process (server restart, API worker)
#   B) The exact input schema
#   C) The exact output schema
#   D) End-to-end integration test on live examples
# ============================================================

# ── A) Loading the model (in a fresh process / API worker) ──
def load_model_artifacts(artifact_dir: str = "model_artifacts") -> dict:
    """
    Call this once at application startup.
    Returns a dict of all artefacts needed by the inference functions.
    """
    return {
        "calibrated_model":    joblib.load(f"{artifact_dir}/model_calibrated.pkl"),
        "raw_model":           joblib.load(f"{artifact_dir}/model_raw.pkl"),
        "shap_explainer":      joblib.load(f"{artifact_dir}/shap_explainer.pkl"),
        "behavioral_features": joblib.load(f"{artifact_dir}/features_behavioral.pkl"),
        "hp_train_medians":    joblib.load(f"{artifact_dir}/hp_train_medians.pkl"),
        "train_medians":       joblib.load(f"{artifact_dir}/train_medians.pkl"),
    }

# ── B) Input schema ──────────────────────────────────────────
REQUIRED_INPUT_FIELDS = [
    "all_clicks",             # int   — total LMS clicks
    "active_days",            # int   — days with any LMS activity
    "access_frequency",       # float — average daily accesses
    "material_clicks",        # float — clicks on course materials
    "quiz_attempts",          # float — total quiz attempts
    "assignment_submissions", # float — assignments submitted
    "total_time_spent",       # int   — minutes on platform
    "procrastination_index",  # float — composite delay score
    "late_submission_count",  # float — number of late submissions
]

# ── C) Core inference function ───────────────────────────────
def predict_student(student_features: dict, artifacts: dict) -> dict:
    """
    Stateless inference function — accepts a flat dict of raw features,
    returns a structured prediction + recommendations dict.

    INPUT  : dict with keys = REQUIRED_INPUT_FIELDS (raw behavioural values)
    OUTPUT : dict with probability, classification, tier, recommendations, counterfactual
    """
    model    = artifacts["calibrated_model"]
    features = artifacts["behavioral_features"]

    # Build raw row
    raw = pd.Series(student_features)

    # Engineer derived features
    train_med = artifacts["train_medians"]
    raw["clicks_per_day"]         = raw["all_clicks"] / (raw["active_days"] + 1)
    raw["time_per_click"]         = raw["total_time_spent"] / (raw["all_clicks"] + 1)
    raw["engagement_consistency"] = raw["active_days"] / (raw["total_time_spent"] / 60 + 1)
    raw["behavioral_risk_score"]  = (
        max(raw["procrastination_index"], 0) * 0.6 +
        raw["late_submission_count"] * 10 * 0.4
    )
    for col in ["all_clicks", "total_time_spent", "active_days"]:
        raw[f"{col}_relative"] = raw[col] / (train_med.get(col, 1) + 1)

    # Align to feature order
    X_student = raw[features].values.reshape(1, -1)
    X_df      = pd.DataFrame(X_student, columns=features)

    probability    = float(model.predict_proba(X_df)[0][1])
    classification = "High Performer" if probability >= 0.5 else "Not High Performer"
    tier           = assign_performance_tier(int(round(probability * 100)))

    # SHAP explanations
    explainer       = artifacts["shap_explainer"]
    shap_vals_raw   = explainer.shap_values(X_df)
    shap_vals_pos   = shap_vals_raw[1] if isinstance(shap_vals_raw, list) else shap_vals_raw
    shap_map        = {f: float(shap_vals_pos[0][i]) for i, f in enumerate(features)}

    # Top 3 negative drivers
    sorted_neg = sorted(
        [(f, v) for f, v in shap_map.items() if v < 0 and f in RECOMMENDATION_MAP],
        key=lambda x: x[1]
    )[:3]

    recs = []
    for feat, shap_val in sorted_neg:
        rec = RECOMMENDATION_MAP[feat].copy()
        rec["feature"]      = feat
        rec["shap_impact"]  = round(shap_val, 4)
        recs.append(rec)

    if not recs:
        recs = [{"icon":"✅","short":"Strong engagement","action":"Keep it up!","why":"Patterns align with high performers.","feature":None,"shap_impact":0.0}]

    return {
        "probability":         round(probability, 4),
        "classification":      classification,
        "tier":                TIER_LABELS[tier],
        "confidence":          "high" if abs(probability - 0.5) > 0.2 else "low",
        "top_negative_drivers": [f for f, _ in sorted_neg],
        "recommendations":     recs,
        "shap_map":            {k: round(v, 4) for k, v in shap_map.items()},
    }


# ── D) Integration test ──────────────────────────────────────
print("=" * 65)
print("INTEGRATION TEST — End-to-End")
print("=" * 65)

# Load artifacts
artifacts = load_model_artifacts("model_artifacts")
print("✅ Artifacts loaded successfully")

# Test student A: low engagement (expected: at-risk)
student_at_risk = {
    "all_clicks": 120, "active_days": 8, "access_frequency": 3.5,
    "material_clicks": 15.0, "quiz_attempts": 1.0, "assignment_submissions": 2.0,
    "total_time_spent": 300, "procrastination_index": 28.0, "late_submission_count": 4.0,
}

# Test student B: high engagement (expected: high performer)
student_high_perf = {
    "all_clicks": 4500, "active_days": 140, "access_frequency": 25.0,
    "material_clicks": 320.0, "quiz_attempts": 14.0, "assignment_submissions": 9.0,
    "total_time_spent": 14000, "procrastination_index": 1.5, "late_submission_count": 0.0,
}

for label, student in [("At-Risk Student", student_at_risk), ("High-Performer Student", student_high_perf)]:
    result = predict_student(student, artifacts)
    print(f"\n--- {label} ---")
    print(f"  Probability    : {result['probability']:.1%}")
    print(f"  Classification : {result['classification']}")
    print(f"  Tier           : {result['tier']}")
    print(f"  Confidence     : {result['confidence']}")
    print(f"  Top drivers    : {result['top_negative_drivers']}")
    if result['recommendations']:
        r = result['recommendations'][0]
        print(f"  Top rec        : {r['icon']} {r['short']}")

print("\n✅ Integration test passed")


INTEGRATION TEST — End-to-End
✅ Artifacts loaded successfully

--- At-Risk Student ---
  Probability    : 0.0%
  Classification : Not High Performer
  Tier           : At Risk ⚠️
  Confidence     : high
  Top drivers    : ['assignment_submissions', 'quiz_attempts', 'active_days']
  Top rec        : 📤 Submit all assignments

--- High-Performer Student ---
  Probability    : 80.6%
  Classification : High Performer
  Tier           : High Performer 🎯
  Confidence     : high
  Top drivers    : ['quiz_attempts', 'all_clicks']
  Top rec        : 📝 Practice quizzes more

✅ Integration test passed


## Section 14 — Live Demo: Recommendations, Counterfactual & Progress Tracking

In [94]:
# ============================================================
# SECTION 14: LIVE DEMO
# ============================================================

# Pick a representative at-risk student from test set
at_risk_indices = [i for i in range(len(X_test))
                   if 0.05 < calibrated_model.predict_proba([X_test.iloc[i]])[0][1] < 0.48
                   and y_test.iloc[i] == 0]

if not at_risk_indices:
    at_risk_indices = (y_test == 0).to_numpy().nonzero()[0].tolist()

demo_idx = at_risk_indices[10]
print(f"Demo student index: {demo_idx}  (true label: {'HP' if y_test.iloc[demo_idx] == 1 else 'Not HP'})")

# ── A) Personalised recommendations ─────────────────────────
rec = generate_recommendations(demo_idx, top_k=3)
display_recommendations(rec)

# ── B) Counterfactual ────────────────────────────────────────
print("\n--- Counterfactual Analysis ---")
cf = find_counterfactual(demo_idx)
print(f"Status:   {cf['status']}")
print(f"Prob:     {cf['original_probability']:.1%}  →  {cf['new_probability']:.1%}  "
      f"(+{cf['probability_gain']:.1%})")
if cf["changes_needed"]:
    print("Suggested changes:")
    for feat, change in cf["changes_needed"].items():
        print(f"  {feat:35s}: {change['from']:.2f}  →  {change['to']:.2f}  ({change['change']:+.2f})")
else:
    print("  No changes needed.")

# ── C) Progress tracking (simulated next-week data) ──────────
print("\n--- Progress Tracking (simulated next-week data) ---")
simulated_next_week = {
    "procrastination_index":   max(0, float(X_test.iloc[demo_idx]["procrastination_index"]) - 10),
    "late_submission_count":   0.0,
    "quiz_attempts":           float(X_test.iloc[demo_idx]["quiz_attempts"]) + 5,
    "active_days":             float(X_test.iloc[demo_idx]["active_days"]) + 15,
}
prog = track_real_progress(demo_idx, simulated_next_week)
print(f"Status : {prog['status']}")
print(f"Prob   : {prog['old_probability']:.1%}  →  {prog['new_probability']:.1%}  "
      f"({prog['improvement']:+.1%})")
print(f"Class  : {prog['old_classification']}  →  {prog['new_classification']}")
print("Feature changes:")
for c in prog["changes"]:
    print(f"  {c['direction']} {c['feature']:35s}: {c['from']}  →  {c['to']}")


Demo student index: 89  (true label: Not HP)
  STUDENT PERFORMANCE REPORT  —  Index 89
  Classification : At Risk ⚠️
  Probability    : 5.6%

📌 Top Areas for Improvement:

  1. 📤  Submit all assignments
     Your value : 3.0  (cohort median: 5.0, 30th percentile)
     ➤  Prioritise completing every assignment, even if imperfect. A submitted assignment always scores better than none.
     💡 Why: Missing assignments compound into significant grade drops.

  2. 📝  Practice quizzes more
     Your value : 0.0  (cohort median: 2.0, 40th percentile)
     ➤  Attempt every available quiz at least twice — first without notes, then with. Focus on wrong answers.
     💡 Why: Active recall through quizzes is one of the most evidence-backed learning strategies.

  3. ⏰  Procrastination is your biggest hurdle
     Your value : -2.33  (cohort median: -0.12, 19th percentile)
     ➤  Break assignments into daily micro-tasks. Use the 2-minute rule: if a task takes under 2 minutes, do it now.
     💡 Why: H